# Manager-question reference evaluation

Build a 14-question reference set from the figures published in notebooks 01 and 03. Expected values are frozen before the checks and recomputed from the local source files.

Each question records expected numbers, acceptable answer requirements, and—where needed—a refusal boundary. This notebook evaluates reference construction, not generated answers.

**Source:** January 2024 feature and label artifacts in `data/labelled/`. The notebook verifies citations and exact values, then writes the reference files and hash sidecars. It makes no external API calls.


In [ ]:
import hashlib, json, re, sys, time

from pathlib import Path

import pandas as pd

from resolveiq.config import REPO_ROOT

LABELED_DIR = REPO_ROOT / "data" / "labelled"
OUT_DIR = REPO_ROOT / "data" / "evaluation"

FEATURES_FILE = "heat_jan2024_event_quiet_features_at_intake.json"
LABELS_FILE = "heat_jan2024_event_quiet_slow_resolution_labels.json"
FEATURES_META = FEATURES_FILE.replace(".json", ".meta.json")
LABELS_META = LABELS_FILE.replace(".json", ".meta.json")

QUESTIONS_VERSION = "v1"
QUESTIONS_FILE = f"manager_questions_{QUESTIONS_VERSION}.jsonl"
EXPECTED_FILE = f"manager_questions_expected_{QUESTIONS_VERSION}.json"
THIS_NOTEBOOK = "notebooks/04_evaluation_design/build_manager_question_eval_set.ipynb"

print(f"repo      {REPO_ROOT}")
print(f"inputs    {LABELED_DIR}")
print(f"outputs   {OUT_DIR}  (created only when this notebook writes)")
print(f"artefacts {QUESTIONS_FILE}   {EXPECTED_FILE}")


repo      C:\Users\mejdi\Documents\resolveiq
inputs    C:\Users\mejdi\Documents\resolveiq\data\labelled
outputs   C:\Users\mejdi\Documents\resolveiq\data\evaluation  (created only when this notebook writes)
artefacts manager_questions_v1.jsonl   manager_questions_expected_v1.json


## What this notebook is allowed to touch

Track 2's argument is that a manager reference evaluation set can be constructed offline: the numbers live in two
local files, so the script needs those files and nothing else. The check below inspects `sys.modules` to verify that
selected HTTP client, model API, and vector-store libraries have not been imported into the kernel.

This module check is a verification safeguard, not an operating-system or network sandbox. The assurance that this
notebook makes no outbound network requests comes from source-code review: all data is loaded locally from
`data/labelled/`, and no network calls exist in any cell.


In [19]:
BANNED_MODULES = ["httpx", "requests", "urllib3", "openai", "anthropic", "google.genai",
                  "google.generativeai", "vertexai", "qdrant_client", "ragas", "langsmith"]

imported = sorted(m for m in BANNED_MODULES if m in sys.modules)
print(f"{len(BANNED_MODULES)} model, vector-store, tracing and HTTP-client libraries checked against sys.modules")
print("present: " + (", ".join(imported) if imported else
                     "none — none of the checked client libraries are loaded in sys.modules"))
assert not imported, f"a client library is already imported here: {imported}"

# the city pull lived in notebook 03; nothing in this notebook should look like one
assert "httpx" not in globals() and "get_settings" not in globals()
print("source review: no outbound network-request code appears in this notebook's cells")


11 model, vector-store, tracing and HTTP-client libraries checked against sys.modules
present: none — none of the checked client libraries are loaded in sys.modules
source review: no outbound network-request code appears in this notebook's cells


## The ten fields every question record carries

The tutor's reference examples stored `inputs.question` plus `outputs.ground_truth` and `reference_context_ids`
(`notebooks/week1/04-evaluation-dataset.ipynb`, cell 22). Same idea, wider, because a number needs more protection
than a sentence:

| field | why it is there |
|---|---|
| `id` | stable handle: `mq-01` … `mq-14` |
| `question` | what a manager would type. Refusal questions must not name the field they are refusing over |
| `question_type` | the coverage claim hangs on this: 2 volume, 3 mix, 2 prevalence, 1 outcome, 1 threshold, 1 limitation, 4 refusal classes |
| `expected_response_type` | `answer`, `answer_with_caveat` or `refuse` — a caveat is not a refusal, and conflating them is how a system learns to dodge |
| `expected_numeric` | the frozen values. Strictly `int`/`float`, so no vague string hides here |
| `reference_record_ids` | the ticket ids a question is *about*, when a handful of records is the answer |
| `acceptable_phrasings` | wording a correct answer must contain. Digit-free: numbers are graded from `expected_numeric`, so a comma or a decimal place cannot fail a right answer |
| `forbidden_phrasings` | wording that makes it wrong even when every number in it is right |
| `expected_refusal_reason` | the boundary being asserted; `""` for answerable questions |
| `source` | artefact, the pandas expression that reproduces it, where the figure was already published, and the original Socrata filter as provenance text |


In [20]:
FIELDS = ["id", "question", "question_type", "expected_response_type", "expected_numeric",
          "reference_record_ids", "acceptable_phrasings", "forbidden_phrasings",
          "expected_refusal_reason", "source"]

# the filter notebook 03 actually used. Kept as text so a reviewer can re-issue it by hand;
# this notebook never sends it.
API_WHERE = ("lower(complaint_type) = 'heat/hot water' "
             "and created_date >= '2024-01-15T00:00:00' and created_date < '2024-01-23T00:00:00'"
             "  /  created_date >= '2024-01-24T00:00:00' and created_date < '2024-01-31T00:00:00'")

EVENT = "event_2024-01-15_22"
QUIET = "quiet_2024-01-24_30"


def src(local_query, published_in, artifact="features + labels"):
    return {"artifact": artifact, "local_query": local_query, "published_in": published_in,
            "api_query": API_WHERE}


## Questions 1–4: volume, closure, and the label's own arithmetic

Questions 1–4 cover volume, closure status, and denominator arithmetic. mq-02 distinguishes total counts across an eight-day event window and a seven-day quiet window from their daily rates: 3.12× by total count and 2.73× per day.

mq-03 is the honesty test for open tickets. 28.7% of closed event tickets were slow; if the 41 still-open ones were
counted as fast the headline would read 28.6. One tenth of a point here, because ~99.8% of a January window has
closed by now — but the same question asked about last month would move a long way, which is why the answer has to
name the denominator it used.


In [21]:
QUESTIONS_A = [
    {
        "id": "mq-01",
        "question": "For Jan 15-22 and Jan 24-30, how many HEAT/HOT WATER requests arrived in each window?",
        "question_type": "volume_change",
        "expected_response_type": "answer",
        "expected_numeric": {"event_created": 22811, "quiet_created": 7315, "total_created": 30126},
        "reference_record_ids": [],
        "acceptable_phrasings": ["requests created", "both windows", "full population"],
        "forbidden_phrasings": ["sample", "estimated", "roughly", "all 311"],
        "expected_refusal_reason": "",
        "source": src("count of rows per window over the features file",
                      "notebook 03 pull gate: '22,811 created / 7,315 created'"),
    },
    {
        "id": "mq-02",
        "question": "How much louder was heat intake during the event window than during the quiet week, as a "
                    "total and per day?",
        "question_type": "volume_change",
        "expected_response_type": "answer_with_caveat",
        "expected_numeric": {"event_days": 8, "quiet_days": 7, "event_per_day": 2851.4, "quiet_per_day": 1045.0,
                             "total_ratio": 3.12, "daily_ratio": 2.73, "busiest_event_day": 18,
                             "busiest_event_n": 3784, "busiest_quiet_day": 30, "busiest_quiet_n": 1365},
        "reference_record_ids": [],
        "acceptable_phrasings": ["per day", "the windows are different lengths", "busiest single day"],
        "forbidden_phrasings": ["same length", "seven days each", "eight days each", "caused by"],
        "expected_refusal_reason": "",
        "source": src("rows per window divided by that window's distinct created dates",
                      "notebook 01: '2,851/day vs 1,045/day ratio 2.73'; busiest day 2024-01-18 = 3784"),
    },
    {
        "id": "mq-03",
        "question": "How many of the requests created in those two windows still have no outcome, and would "
                    "counting them as resolved quickly change the headline?",
        "question_type": "outcome_integrity",
        "expected_response_type": "answer",
        "expected_numeric": {"created_total": 30126, "labelled_closed": 30079, "unlabelled_open": 47,
                             "event_unlabelled": 41, "quiet_unlabelled": 6, "event_pct_closed_basis": 28.7,
                             "event_pct_if_open_fast": 28.6, "quiet_pct_closed_basis": 10.0,
                             "quiet_pct_if_open_fast": 10.0},
        "reference_record_ids": [60027107, 60029622, 60030710, 60033080, 60033105, 60042161, 60043463, 60047982,
                                 60049212, 60049389, 60049394, 60050705, 60050743, 60050951, 60052234, 60053994,
                                 60055532, 60057182, 60058466, 60058565, 60060120, 60060223, 60061962, 60062174,
                                 60063994, 60064028, 60065399, 60067202, 60067557, 60074377, 60079114, 60080446,
                                 60084135, 60085131, 60087393, 60089250, 60093337, 60094640, 60098975, 60105614,
                                 60109933, 60153697, 60160814, 60161699, 60165390, 60178397, 60186575],
        "acceptable_phrasings": ["no label", "closed tickets only", "still open"],
        "forbidden_phrasings": ["failed", "resolved quickly", "treated as fast", "assumed closed"],
        "expected_refusal_reason": "",
        "source": src("features keys minus labels keys; slow share over closed and over created",
                      "notebook 03 open-ticket table: 41 / 6 open, 28.7 against 28.6"),
    },
    {
        "id": "mq-04",
        "question": "What share of closed heat requests took longer than the slow threshold in each window, and "
                    "what is the share over both together?",
        "question_type": "prevalence",
        "expected_response_type": "answer_with_caveat",
        "expected_numeric": {"event_closed": 22770, "event_slow": 6526, "event_slow_pct": 28.7,
                             "quiet_closed": 7309, "quiet_slow": 731, "quiet_slow_pct": 10.0,
                             "pooled_closed": 30079, "pooled_slow": 7257, "pooled_slow_pct": 24.1,
                             "threshold_hours": 62.349333},
        "reference_record_ids": [],
        "acceptable_phrasings": ["of closed tickets", "by construction", "threshold"],
        "forbidden_phrasings": ["sla", "breach", "city standard", "target", "service level"],
        "expected_refusal_reason": "",
        "source": src("mean of slow over closed tickets, per window and pooled",
                      "notebook 03 label table: 6,526 slow (28.7%) and 731 (10.0%)"),
    },
]
print(f"{len(QUESTIONS_A)} questions authored, "
      f"{sum(len(q['expected_numeric']) for q in QUESTIONS_A)} frozen values so far")


4 questions authored, 32 frozen values so far


## Questions 5–9: the threshold's own history, and where the mix does and does not matter

Questions 5–9 cover threshold provenance, closed-ticket rates, and borough comparisons. The 62.3-hour threshold is the quiet-window p90 and is not a service standard. Borough rates require denominators; the smaller Staten Island sample is especially important to report.

mq-09 is the borough question, and it is deliberately hard to answer well. Staten Island's 39.4% is the highest
observed share and sits on 289 closed tickets; the Bronx's 12.6% sits on 10,651. So the expected answer needs its
denominators in the same breath as its percentages, plus the standing note that borough is a proxy which has not
passed a fairness review.


In [22]:
QUESTIONS_B = [
    {
        "id": "mq-05",
        "question": "Where did the 62.3-hour threshold come from, and is it a city service standard?",
        "question_type": "threshold_provenance",
        "expected_response_type": "answer_with_caveat",
        "expected_numeric": {"threshold_hours": 62.349333, "threshold_displayed": 62.3,
                             "fitted_on_closed": 7309},
        "reference_record_ids": [],
        "acceptable_phrasings": ["ninetieth percentile", "quiet week", "not a published standard"],
        "forbidden_phrasings": ["sla", "city standard", "policy", "mandated", "target of"],
        "expected_refusal_reason": "",
        "source": src("the label file carries the threshold in every row; notebook 03 fitted it from the quiet week",
                      "notebook 03 threshold cell: 'derived threshold 62.349333 h -> displayed 62.3 h'"),
    },
    {
        "id": "mq-06",
        "question": "What was the symptom mix in each window? Give the counts and shares for NO HEAT, "
                    "NO HEAT AND NO HOT WATER and NO HOT WATER among closed requests.",
        "question_type": "category_mix",
        "expected_response_type": "answer",
        "expected_numeric": {"event_no_heat_n": 15307, "event_no_heat_no_water_n": 6218,
                             "event_no_hot_water_n": 1245, "event_no_heat_pct": 67.2,
                             "event_no_heat_no_water_pct": 27.3, "event_no_hot_water_pct": 5.5,
                             "quiet_no_heat_n": 3915, "quiet_no_heat_no_water_n": 2491,
                             "quiet_no_hot_water_n": 903, "quiet_no_heat_pct": 53.6,
                             "quiet_no_heat_no_water_pct": 34.1, "quiet_no_hot_water_pct": 12.4},
        "reference_record_ids": [],
        "acceptable_phrasings": ["of closed requests", "each window", "the three symptom labels"],
        "forbidden_phrasings": ["of all requests", "pooled", "caused"],
        "expected_refusal_reason": "",
        "source": src("value_counts of descriptor_2 within each window, closed tickets only",
                      "notebook 01 per-window table: 'n=15,307 / 6,218 / 1,245' and 'n=3,915 / 2,491 / 903'"),
    },
    {
        "id": "mq-07",
        "question": "Across both windows, how do ENTIRE BUILDING and APARTMENT ONLY compare on volume and on the "
                    "slow share?",
        "question_type": "category_mix",
        "expected_response_type": "answer",
        "expected_numeric": {"building_n": 18774, "building_slow_pct": 22.8, "apartment_n": 11305,
                             "apartment_slow_pct": 26.3, "spread_pp": 3.5},
        "reference_record_ids": [],
        "acceptable_phrasings": ["closed tickets", "both windows combined", "descriptive"],
        "forbidden_phrasings": ["predicts", "risk of", "accuracy", "proves"],
        "expected_refusal_reason": "",
        "source": src("groupby descriptor over the pooled closed view",
                      "notebook 03 signal table: ENTIRE BUILDING 18,774 at 22.8%, APARTMENT ONLY 11,305 at 26.3%"),
    },
    {
        "id": "mq-08",
        "question": "Which intake channel carries the most heat requests, and does the slow share differ by "
                    "channel?",
        "question_type": "category_mix",
        "expected_response_type": "answer",
        "expected_numeric": {"online_n": 14424, "online_slow_pct": 23.8, "phone_n": 10402,
                             "phone_slow_pct": 27.2, "mobile_n": 5253, "mobile_slow_pct": 19.0,
                             "spread_pp": 8.2},
        "reference_record_ids": [],
        "acceptable_phrasings": ["closed tickets", "share of volume", "channel"],
        "forbidden_phrasings": ["predicts", "because callers", "accuracy"],
        "expected_refusal_reason": "",
        "source": src("groupby open_data_channel_type over the pooled closed view",
                      "notebook 03 signal table: ONLINE 14,424 / PHONE 10,402 / MOBILE 5,253"),
    },
    {
        "id": "mq-09",
        "question": "Which borough had the highest observed slow-ticket share, and what was it standing on?",
        "question_type": "prevalence",
        "expected_response_type": "answer_with_caveat",
        "expected_numeric": {"highest_borough_pct": 39.4, "highest_borough_n": 289, "lowest_borough_pct": 12.6,
                             "lowest_borough_n": 10651, "spread_pp": 26.8, "borough_levels": 5},
        "reference_record_ids": [],
        "acceptable_phrasings": ["closed tickets behind it", "small denominator", "needs a fairness review"],
        "forbidden_phrasings": ["worst borough", "because of where", "certainly", "discriminat"],
        "expected_refusal_reason": "",
        "source": src("groupby borough over the pooled closed view, ranked by slow share",
                      "notebook 03 signal table: STATEN ISLAND 289 at 39.4%, BRONX 10,651 at 12.6%"),
    },
]
print(f"{len(QUESTIONS_B)} questions authored, "
      f"{sum(len(q['expected_numeric']) for q in QUESTIONS_B)} frozen values in this block")


5 questions authored, 33 frozen values in this block


## Questions 10–14: four refusals, and the one answer that only needs a caveat

Questions 10–14 test four refusal boundaries and one answer-with-caveat case. mq-10 lacks weather evidence; mq-14 exceeds the population covered by these files. The latter is a scope refusal and must not name a missing field.

mq-12 and mq-13 are the two field-boundary cases: precise geography (never fetched, never stored) and target outcome
leakage. A ticket's own closure and resolution fields describe what happened to it after intake, so they can never be
inputs to score that ticket. In contrast, retrieving outcomes from prior cases that were already resolved before scoring
time as contextual evidence is permitted and expected in ResolveIQ. Their questions are phrased the way a manager would
phrase them, without naming a column, so the system has to recognise the boundary itself.

mq-11 is the one that must **not** be a refusal. Friday really is the highest observed weekday share and Tuesday the
lowest, and the honest answer is those numbers plus the caveat: the 15 dates across two differently sized windows are
too limited to establish a recurring weekday pattern. Earlier drafts filed this as a refusal; it is an answer with
a caveat, and grading it as a refusal would train the system to withhold numbers it is entitled to give.


In [23]:
QUESTIONS_C = [
    {
        "id": "mq-10",
        "question": "Did the cold weather cause the longer resolutions during the event window?",
        "question_type": "refusal_missing_evidence",
        "expected_response_type": "refuse",
        "expected_numeric": {},
        "reference_record_ids": [],
        "acceptable_phrasings": ["no temperature", "volume was measured", "not established"],
        "forbidden_phrasings": ["yes, snow", "caused by", "the data shows that colder", "391"],
        "expected_refusal_reason": "Nothing in the source records temperature, weather or conditions, so no causal "
                                   "claim about it is supported. The event window is defined by intake volume, and a "
                                   "movement in volume alongside a movement in delay is not evidence of cause.",
        "source": src("no recompute is expected; the absence of a weather field is checked against the field list "
                      "the metadata read published", "notebook 03 metadata read: the 48 source field names"),
    },
    {
        "id": "mq-11",
        "question": "Which weekdays had the highest and lowest observed slow shares, and what caveat is needed "
                    "before treating that as a weekday pattern?",
        "question_type": "limitation",
        "expected_response_type": "answer_with_caveat",
        "expected_numeric": {"friday_slow_pct": 42.8, "friday_n": 3101, "tuesday_slow_pct": 6.4,
                             "tuesday_n": 4031, "spread_pp": 36.4, "observed_dates": 15},
        "reference_record_ids": [],
        "acceptable_phrasings": ["closed tickets", "few observed dates", "entangled with window membership"],
        "forbidden_phrasings": ["weekends are worse because", "predicts", "proves", "different weekdays"],
        "expected_refusal_reason": "",
        "source": src("groupby day name over the pooled closed view, ranked by slow share; distinct dates in view",
                      "notebook 03 signal spread: created_day_of_week 36.4 pp, Friday 42.8%, Tuesday 6.4%"),
    },
    {
        "id": "mq-12",
        "question": "Which corner or address had the slowest heat requests in the event window?",
        "question_type": "refusal_precise_geography",
        "expected_response_type": "refuse",
        "expected_numeric": {},
        "reference_record_ids": [],
        "acceptable_phrasings": ["never stored", "coarse geography only", "borough needs a fairness review"],
        "forbidden_phrasings": ["the corner of", "zip code", "at 123", "between avenues"],
        "expected_refusal_reason": "Precise geography is not present in these files by rule: incident_zip, "
                                   "street_name, latitude, longitude, cross_street_1, cross_street_2, "
                                   "intersection_street_1 and intersection_street_2 are forbidden fields that were "
                                   "never fetched, so nothing finer than the coarse service geography in hand can be "
                                   "answered.",
        "source": src("no recompute is expected; the reason is checked against the sidecar's forbidden field list",
                      "notebook 03 fail-closed field policy: 39 forbidden fields"),
    },
    {
        "id": "mq-13",
        "question": "Can we predict slow-resolution risk for new incoming tickets by checking when they were closed or "
                    "what action was taken?",
        "question_type": "refusal_outcome_field",
        "expected_response_type": "refuse",
        "expected_numeric": {},
        "reference_record_ids": [],
        "acceptable_phrasings": ["target leakage", "written after intake", "historical cases as evidence", "forbidden as model inputs"],
        "forbidden_phrasings": ["yes, use closed_date", "accuracy", "guarantee", "historical outcomes cannot be retrieved"],
        "expected_refusal_reason": "A ticket's own post-intake outcome fields (closed_date, status, resolution_description, "
                                   "resolution_action_updated_date, due_date) are written after the request arrives and are strictly "
                                   "forbidden as scoring inputs for that ticket; using them is target leakage. In contrast, retrieving "
                                   "outcomes from prior cases already resolved before scoring time as historical evidence is permitted, "
                                   "but the target ticket's own outcome can never be an input.",
        "source": src("no recompute is expected; the reason is checked against the sidecar's forbidden field list",
                      "notebook 03 leak guard: closure fields excluded from the features file"),
    },
    {
        "id": "mq-14",
        "question": "What is the slow-resolution rate across all of 311 citywide this year?",
        "question_type": "refusal_scope",
        "expected_response_type": "refuse",
        "expected_numeric": {},
        "reference_record_ids": [],
        "acceptable_phrasings": ["one complaint type", "two January windows", "outside these files"],
        "forbidden_phrasings": ["citywide rate", "all complaints", "every agency", "this year"],
        "expected_refusal_reason": "The question is answerable in principle but not from these artefacts: they hold "
                                   "HEAT/HOT WATER requests created in two January 2024 windows, so any citywide or "
                                   "full-year figure would be an invented extrapolation. This limit is about scope, "
                                   "not about a column that happens to be missing.",
        "source": src("no recompute is expected; the scope is checked against the sidecar's windows and row counts",
                      "notebook 03 windows block: 22,811 and 7,315 rows, one complaint type"),
    },
]
QUESTIONS = QUESTIONS_A + QUESTIONS_B + QUESTIONS_C
assert [q["id"] for q in QUESTIONS] == [f"mq-{n:02d}" for n in range(1, 15)]
print(f"{len(QUESTIONS)} questions authored and frozen before any recompute: "
      f"{sum(len(q['expected_numeric']) for q in QUESTIONS)} expected values, "
      f"{sum(1 for q in QUESTIONS if q['expected_response_type'] == 'refuse')} refusals")


14 questions authored and frozen before any recompute: 71 expected values, 4 refusals


## Gate 1: is this a well-formed question set?

Gate 1 checks record schema, unique IDs, category counts, refusal structure, and phrasing rules. Numeric answers are checked from `expected_numeric`, independent of formatting. It does not read source data.


In [24]:
REQUIRED_TYPES = {"volume_change": 2, "outcome_integrity": 1, "prevalence": 2,
                  "threshold_provenance": 1, "category_mix": 3, "limitation": 1,
                  "refusal_missing_evidence": 1, "refusal_precise_geography": 1,
                  "refusal_outcome_field": 1, "refusal_scope": 1}
REFUSALS = [q for q in QUESTIONS if q["expected_response_type"] == "refuse"]
ANSWERED = [q for q in QUESTIONS if q["expected_response_type"] != "refuse"]
ANSWER_ID = {q["id"]: q for q in QUESTIONS}
types = [q["question_type"] for q in QUESTIONS]
rows = []


def check(name, ok, detail):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


check("fourteen questions", len(QUESTIONS) == 14, f"{len(QUESTIONS)} authored, ids "
      f"{QUESTIONS[0]['id']}..{QUESTIONS[-1]['id']}")
check("every record carries exactly the ten fields, in order", all(list(q) == FIELDS for q in QUESTIONS),
      f"{len(FIELDS)} fields x {len(QUESTIONS)} records")
check("questions are distinct", len({q["question"].lower() for q in QUESTIONS}) == 14, "no repeated question")
check("question-type counts match the design", sorted(types) == sorted(t for t, n in REQUIRED_TYPES.items()
                                                                      for _ in range(n)),
      "  ".join(f"{t} {n}" for t, n in sorted(REQUIRED_TYPES.items())))
check("every answerable question carries frozen numbers", all(q["expected_numeric"] for q in ANSWERED),
      f"{sum(len(q['expected_numeric']) for q in ANSWERED)} values over {len(ANSWERED)} answerable questions")
check("no refusal question carries a number", not any(q["expected_numeric"] for q in REFUSALS),
      f"{len(REFUSALS)} refusals, 0 expected values")
check("no refusal question names a field it refuses over", all(
    not re.search(r"\b(closed_date|incident_zip|street_name|latitude|longitude|status|due_date|resolution_\w+)\b",
                  q["question"]) for q in REFUSALS), "each refusal is phrased the way a manager would phrase it")
check("expected values are only int or float", all(isinstance(v, (int, float)) and not isinstance(v, bool)
                                                   for q in QUESTIONS for v in q["expected_numeric"].values()),
      f"{sum(len(q['expected_numeric']) for q in QUESTIONS)} values, no strings, so nothing vague hides here")
check("acceptable phrasings are digit-free", not any(ch.isdigit() for q in QUESTIONS
                                                    for p in q["acceptable_phrasings"] for ch in p),
      "numbers are graded from expected_numeric, not from wording")
check("acceptable and forbidden wording never cancel", all(
    not {p.lower() for p in q["acceptable_phrasings"]} & {p.lower() for p in q["forbidden_phrasings"]}
    for q in QUESTIONS), "no phrase asked for in the same breath as it is banned")
check("a caveat is not a refusal", all(q["expected_refusal_reason"] == "" for q in ANSWERED)
      and all(q["expected_refusal_reason"] for q in REFUSALS),
      f"{sum(1 for q in QUESTIONS if q['expected_response_type'] == 'answer_with_caveat')} answers carry a caveat, "
      f"{len(REFUSALS)} refusals carry a reason")
check("reference ids are only used where records are the answer",
      sum(len(q["reference_record_ids"]) for q in QUESTIONS) <= 50,
      f"{sum(1 for q in QUESTIONS if q['reference_record_ids'])} question names records, "
      f"{sum(len(q['reference_record_ids']) for q in QUESTIONS)} ids total")

gate1 = pd.DataFrame(rows)
print(gate1.to_string(index=False))
fails = int((gate1["result"] == "FAIL").sum())
print(f"\n{fails} of {len(gate1)} failed")
assert fails == 0, "the question set is not well formed, so nothing downstream can trust it"


                                                   check result                                                                                                                                                                                                      detail
                                      fourteen questions   PASS                                                                                                                                                                               14 authored, ids mq-01..mq-14
   every record carries exactly the ten fields, in order   PASS                                                                                                                                                                                      10 fields x 14 records
                                  questions are distinct   PASS                                                                                                                                     

## Where each frozen number was already published

Each expected value cites the notebook cell that published it. This check opens the stored output and verifies the cited figure and field references.

The one exception is `mq-03`'s list of 47 ticket ids: it came from these artefacts themselves (features keys minus
labels keys), not from an independent pull. It pins *which* tickets the answer must be about; it is not a second
measurement of anything.


In [25]:
NB01 = "notebooks/01_data_exploration/nyc311_snapshot_profile.ipynb"
NB03 = "notebooks/03_target_definition/define_slow_resolution_label.ipynb"

PROVENANCE = {
    "mq-01": {"artifact": NB03, "cell": 7, "quoted": "22,811 created"},
    "mq-02": {"artifact": NB01, "cell": 33, "quoted": "2.73",
              "also": {"artifact": NB01, "cell": 30, "quoted": "3784"}},
    "mq-03": {"artifact": NB03, "cell": 15, "quoted": "28.6",
              "note": "the 47 ticket ids come from these artefacts, not an independent pull"},
    "mq-04": {"artifact": NB03, "cell": 13, "quoted": "6,526"},
    "mq-05": {"artifact": NB03, "cell": 13, "quoted": "62.349333"},
    "mq-06": {"artifact": NB01, "cell": 36, "quoted": "n=15,307",
              "also": {"artifact": NB01, "cell": 36, "quoted": "n= 3,915"}},
    "mq-07": {"artifact": NB03, "cell": 21, "quoted": "18774"},
    "mq-08": {"artifact": NB03, "cell": 21, "quoted": "14424"},
    "mq-09": {"artifact": NB03, "cell": 21, "quoted": "39.4"},
    "mq-10": {"artifact": NB03, "cell": 3, "absent_terms": ["temperature", "weather", "forecast"]},
    "mq-11": {"artifact": NB03, "cell": 21, "quoted": "42.8"},
    "mq-12": {"artifact": f"data/labelled/{FEATURES_META}", "key": "forbidden_fields", "quoted": "incident_zip"},
    "mq-13": {"artifact": f"data/labelled/{FEATURES_META}", "key": "forbidden_fields", "quoted": "closed_date"},
    "mq-14": {"artifact": f"data/labelled/{FEATURES_META}", "key": "windows", "quoted": "22811"},
}

_nb_cache, _side_cache = {}, {}


def published(entry):
    # a figure's citation: a finished run's stored output, or a value from a sidecar
    path = REPO_ROOT / entry["artifact"]
    assert path.exists(), f"provenance points at a file that does not exist: {entry['artifact']}"
    if "cell" in entry:
        if entry["artifact"] not in _nb_cache:
            _nb_cache[entry["artifact"]] = json.loads(path.read_text(encoding="utf-8"))["cells"]
        cells = _nb_cache[entry["artifact"]]
        assert entry["cell"] < len(cells), f"{path.name} has no cell {entry['cell']} (it has {len(cells)})"
        assert cells[entry["cell"]]["cell_type"] == "code", f"{path.name} cell {entry['cell']} is not a run cell"
        out = "".join("".join(o.get("text", "")) for o in cells[entry["cell"]].get("outputs", []))
        assert out.strip(), f"{path.name} cell {entry['cell']} has no stored output to cite"
        return f"{path.name} cell {entry['cell']}", out
    if entry["artifact"] not in _side_cache:
        _side_cache[entry["artifact"]] = json.loads(path.read_text(encoding="utf-8"))
    return (f"{path.name} ['{entry['key']}']",
            json.dumps(_side_cache[entry["artifact"]][entry["key"]], default=str))


print(f"{len(PROVENANCE)} provenance entries checked against the files that published them")
for q in QUESTIONS:
    entry = PROVENANCE[q["id"]]
    where, text = published(entry)
    marks = []
    if "quoted" in entry:
        assert entry["quoted"] in text, f"{q['id']}: '{entry['quoted']}' is not in {where}"
        marks.append(f"'{entry['quoted']}'")
    for term in entry.get("absent_terms", []):
        assert term not in text.lower(), f"{q['id']}: the source does have a '{term}' field, per {where}"
        marks.append(f"no '{term}'")
    if "also" in entry:
        where2, text2 = published(entry["also"])
        assert entry["also"]["quoted"] in text2, f"{q['id']}: '{entry['also']['quoted']}' not in {where2}"
        marks.append(f"'{entry['also']['quoted']}'")
        where = f"{where} + {where2}"
    print(f"  {q['id']}  {marks[0]:<26} {where}")
assert set(PROVENANCE) == {q["id"] for q in QUESTIONS}, "every question must cite where its figure came from"


14 provenance entries checked against the files that published them
  mq-01  '22,811 created'           define_slow_resolution_label.ipynb cell 7
  mq-02  '2.73'                     nyc311_snapshot_profile.ipynb cell 33 + nyc311_snapshot_profile.ipynb cell 30
  mq-03  '28.6'                     define_slow_resolution_label.ipynb cell 15
  mq-04  '6,526'                    define_slow_resolution_label.ipynb cell 13
  mq-05  '62.349333'                define_slow_resolution_label.ipynb cell 13
  mq-06  'n=15,307'                 nyc311_snapshot_profile.ipynb cell 36 + nyc311_snapshot_profile.ipynb cell 36
  mq-07  '18774'                    define_slow_resolution_label.ipynb cell 21
  mq-08  '14424'                    define_slow_resolution_label.ipynb cell 21
  mq-09  '39.4'                     define_slow_resolution_label.ipynb cell 21
  mq-10  no 'temperature'           define_slow_resolution_label.ipynb cell 3
  mq-11  '42.8'                     define_slow_resolution_label.ipynb cel

## Read the two artefacts — no request is sent

Verify each file's SHA-256 and row count against notebook 03's sidecar before using it. Rebuild the source field list from the same metadata.

The field list is checked against the source metadata rather than typed from memory.


In [26]:
feat_side = json.loads((LABELED_DIR / FEATURES_META).read_text(encoding="utf-8"))
lab_side = json.loads((LABELED_DIR / LABELS_META).read_text(encoding="utf-8"))
rows = []


def check(name, ok, detail):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


for label, side, name in (("features", feat_side, FEATURES_FILE), ("labels", lab_side, LABELS_FILE)):
    body = (LABELED_DIR / name).read_bytes()
    check(f"the {label} file is the file its sidecar describes",
          hashlib.sha256(body).hexdigest() == side["artifact_sha256"] and len(body) == side["artifact_bytes"],
          f"{len(body):,} bytes   sha256 {side['artifact_sha256'][:16]}")

ALL_SOURCE = set(feat_side["intake_fields"]) | set(feat_side["forbidden_fields"]) | {"unique_key", "created_date"}
FORBIDDEN = set(feat_side["forbidden_fields"])
INTAKE = list(feat_side["intake_fields"])
check("the source field list rebuilds to the city's 48 columns", len(ALL_SOURCE) == 48,
      f"{len(feat_side['intake_fields'])} intake + {len(FORBIDDEN)} forbidden + 2 audit")
check("no weather or temperature field exists", not {"temperature", "weather", "forecast"} & ALL_SOURCE,
      "mq-10's refusal rests on the field list, not on a sentence written here")
check("closure and outcome fields are all forbidden",
      {"closed_date", "status", "resolution_description", "resolution_action_updated_date", "due_date"} <= FORBIDDEN,
      "mq-13's boundary is the published policy")
check("precise geography fields are all forbidden",
      {"incident_zip", "street_name", "latitude", "longitude", "cross_street_1", "cross_street_2",
       "intersection_street_1", "intersection_street_2"} <= FORBIDDEN, "mq-12's boundary is the published policy")

feat = pd.read_json(LABELED_DIR / FEATURES_FILE, dtype={"unique_key": "int64"})
lab = pd.read_json(LABELED_DIR / LABELS_FILE, dtype={"unique_key": "int64"})
feat["created_date"] = pd.to_datetime(feat["created_date"])
check("row counts match what the sidecars recorded",
      len(feat) == feat_side["rows"] and len(lab) == lab_side["rows"], f"{len(feat):,} / {len(lab):,}")
check("the features file holds no forbidden column", not set(feat.columns) & FORBIDDEN,
      f"{len(feat.columns)} columns: {', '.join(feat.columns)}")
check("every label belongs to a features row", set(lab.unique_key) <= set(feat.unique_key),
      f"{len(set(feat.unique_key) - set(lab.unique_key))} features rows carry no label")
check("the threshold is one exact value in every row", lab.label_threshold_hours.nunique() == 1,
      f"{float(lab.label_threshold_hours.iloc[0]):.6f} h, unchanged from notebook 03")

open_tickets = feat.loc[~feat.unique_key.isin(lab.unique_key)]
closed = feat.merge(lab, on="unique_key", how="inner")
closed["slow"] = closed["slow_resolution"].astype(bool)
check("closed + open = created, with nothing in both", len(closed) + len(open_tickets) == len(feat),
      f"{len(closed):,} closed + {len(open_tickets):,} open = {len(feat):,}")

gate = pd.DataFrame(rows)
print(gate.to_string(index=False))
fails = int((gate["result"] == "FAIL").sum())
print(f"\n{fails} of {len(gate)} failed   |   requests sent by this notebook: 0")
assert fails == 0, "the inputs are not what the sidecars describe, so no expected value below means anything"


                                                  check result                                                                                                                                                                                                detail
    the features file is the file its sidecar describes   PASS                                                                                                                                                            14,221,430 bytes   sha256 8d26ccbb40f73d14
      the labels file is the file its sidecar describes   PASS                                                                                                                                                             3,632,305 bytes   sha256 67caa358422303cc
the source field list rebuilds to the city's 48 columns   PASS                                                                                                                                                           

## Recompute every expected value from those files

Recompute the reference values from the two local artifacts using pandas. Shares are rounded to one decimal place and the threshold to six.


In [27]:
def pct(n, d):
    return round(100 * n / d, 1)


def ranked(frame, col):
    g = frame.groupby(col)["slow"].agg(["size", "sum"]).sort_values("size", ascending=False)
    return {lvl: (int(r["size"]), int(r["sum"]), pct(r["sum"], r["size"])) for lvl, r in g.iterrows()}


ev_c, qu_c = closed[closed.window == EVENT], closed[closed.window == QUIET]
ev_all, qu_all = feat[feat.window == EVENT], feat[feat.window == QUIET]
ev_open = open_tickets[open_tickets.window == EVENT]
qu_open = open_tickets[open_tickets.window == QUIET]
ev_days = ev_all.groupby(ev_all.created_date.dt.date).size()
qu_days = qu_all.groupby(qu_all.created_date.dt.date).size()
ev_rate, qu_rate = len(ev_all) / len(ev_days), len(qu_all) / len(qu_days)
borough = ranked(closed, "borough")
channel = ranked(closed, "open_data_channel_type")
descriptor = ranked(closed, "descriptor")
weekday = ranked(closed.assign(name=closed.created_date.dt.day_name()), "name")
symptom_ev, symptom_qu = ev_c.descriptor_2.value_counts(), qu_c.descriptor_2.value_counts()
threshold_h = float(lab.label_threshold_hours.iloc[0])
hi = max(borough, key=lambda k: borough[k][2])
lo = min(borough, key=lambda k: borough[k][2])
NO_HEAT, BOTH, NO_WATER = "NO HEAT", "NO HEAT AND NO HOT WATER", "NO HOT WATER"

RECOMPUTED = {
    "mq-01": {"event_created": len(ev_all), "quiet_created": len(qu_all), "total_created": len(feat)},
    "mq-02": {"event_days": len(ev_days), "quiet_days": len(qu_days),
              "event_per_day": round(ev_rate, 1), "quiet_per_day": round(qu_rate, 1),
              "total_ratio": round(len(ev_all) / len(qu_all), 2), "daily_ratio": round(ev_rate / qu_rate, 2),
              "busiest_event_day": int(ev_days.idxmax().day), "busiest_event_n": int(ev_days.max()),
              "busiest_quiet_day": int(qu_days.idxmax().day), "busiest_quiet_n": int(qu_days.max())},
    "mq-03": {"created_total": len(feat), "labelled_closed": len(closed), "unlabelled_open": len(open_tickets),
              "event_unlabelled": len(ev_open), "quiet_unlabelled": len(qu_open),
              "event_pct_closed_basis": pct(ev_c.slow.sum(), len(ev_c)),
              "event_pct_if_open_fast": pct(ev_c.slow.sum(), len(ev_all)),
              "quiet_pct_closed_basis": pct(qu_c.slow.sum(), len(qu_c)),
              "quiet_pct_if_open_fast": pct(qu_c.slow.sum(), len(qu_all))},
    "mq-04": {"event_closed": len(ev_c), "event_slow": int(ev_c.slow.sum()),
              "event_slow_pct": pct(ev_c.slow.sum(), len(ev_c)),
              "quiet_closed": len(qu_c), "quiet_slow": int(qu_c.slow.sum()),
              "quiet_slow_pct": pct(qu_c.slow.sum(), len(qu_c)),
              "pooled_closed": len(closed), "pooled_slow": int(closed.slow.sum()),
              "pooled_slow_pct": pct(closed.slow.sum(), len(closed)), "threshold_hours": threshold_h},
    "mq-05": {"threshold_hours": threshold_h, "threshold_displayed": round(threshold_h, 1),
              "fitted_on_closed": len(qu_c)},
    "mq-06": {"event_no_heat_n": int(symptom_ev[NO_HEAT]), "event_no_heat_no_water_n": int(symptom_ev[BOTH]),
              "event_no_hot_water_n": int(symptom_ev[NO_WATER]), "event_no_heat_pct": pct(symptom_ev[NO_HEAT], len(ev_c)),
              "event_no_heat_no_water_pct": pct(symptom_ev[BOTH], len(ev_c)),
              "event_no_hot_water_pct": pct(symptom_ev[NO_WATER], len(ev_c)),
              "quiet_no_heat_n": int(symptom_qu[NO_HEAT]), "quiet_no_heat_no_water_n": int(symptom_qu[BOTH]),
              "quiet_no_hot_water_n": int(symptom_qu[NO_WATER]), "quiet_no_heat_pct": pct(symptom_qu[NO_HEAT], len(qu_c)),
              "quiet_no_heat_no_water_pct": pct(symptom_qu[BOTH], len(qu_c)),
              "quiet_no_hot_water_pct": pct(symptom_qu[NO_WATER], len(qu_c))},
    "mq-07": {"building_n": descriptor["ENTIRE BUILDING"][0], "building_slow_pct": descriptor["ENTIRE BUILDING"][2],
              "apartment_n": descriptor["APARTMENT ONLY"][0], "apartment_slow_pct": descriptor["APARTMENT ONLY"][2],
              "spread_pp": round(abs(descriptor["ENTIRE BUILDING"][2] - descriptor["APARTMENT ONLY"][2]), 1)},
    "mq-08": {"online_n": channel["ONLINE"][0], "online_slow_pct": channel["ONLINE"][2],
              "phone_n": channel["PHONE"][0], "phone_slow_pct": channel["PHONE"][2],
              "mobile_n": channel["MOBILE"][0], "mobile_slow_pct": channel["MOBILE"][2],
              "spread_pp": round(max(v[2] for v in channel.values()) - min(v[2] for v in channel.values()), 1)},
    "mq-09": {"highest_borough_pct": borough[hi][2], "highest_borough_n": borough[hi][0],
              "lowest_borough_pct": borough[lo][2], "lowest_borough_n": borough[lo][0],
              "spread_pp": round(borough[hi][2] - borough[lo][2], 1), "borough_levels": len(borough)},
    "mq-11": {"friday_slow_pct": weekday["Friday"][2], "friday_n": weekday["Friday"][0],
              "tuesday_slow_pct": weekday["Tuesday"][2], "tuesday_n": weekday["Tuesday"][0],
              "spread_pp": round(max(v[2] for v in weekday.values()) - min(v[2] for v in weekday.values()), 1),
              "observed_dates": len(feat.created_date.dt.date.unique())},
}
print(f"{sum(len(v) for v in RECOMPUTED.values())} values recomputed from the two artefacts, "
      f"across {len(RECOMPUTED)} questions")
print("borough slow shares, highest to lowest: " +
      "  ".join(f"{k.title()} {v[2]}% (n={v[0]:,})" for k, v in sorted(borough.items(), key=lambda kv: -kv[1][2])))


71 values recomputed from the two artefacts, across 10 questions
borough slow shares, highest to lowest: Staten Island 39.4% (n=289)  Manhattan 32.8% (n=6,692)  Brooklyn 29.0% (n=7,821)  Queens 28.9% (n=4,626)  Bronx 12.6% (n=10,651)


## Gate 2: the frozen expectations against the files on disk

Compare every frozen value at its published precision: integers exactly, shares to one decimal place, and the threshold to six decimals.

Notebook 01 counted closed tickets in a pull of its own; notebook 03 labelled a pull it made for another purpose. The
two agreeing is therefore a cross-check between two artefacts, not a value being compared with itself.


In [28]:
def decimals(v):
    return len(str(v).split(".")[1]) if isinstance(v, float) and "." in str(v) else 0


rows = []


def check(name, ok, detail):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


agree, mismatches = [], []
for q in QUESTIONS:
    exp, got = q["expected_numeric"], RECOMPUTED.get(q["id"], {})
    if not exp:
        agree.append({"id": q["id"], "values": 0, "exact_matches": 0, "first_mismatch": ""})
        check(f"{q['id']} refusal", set(got) == set(), "no number authored, none expected")
        continue
    off = [(k, exp[k], got[k]) for k in exp if k in got
           and round(got[k], decimals(exp[k])) != round(exp[k], decimals(exp[k]))]
    missing = [k for k in exp if k not in got] + [k for k in got if k not in exp]
    check(f"{q['id']} — {len(exp)} values", not missing and not off,
          "all exact" if not missing and not off else "; ".join(
              f"{k}: frozen {exp.get(k, 'absent')} vs file {got.get(k, 'absent')}" for k in missing + [o[0] for o in off]))
    agree.append({"id": q["id"], "values": len(exp), "exact_matches": len(exp) - len(off),
                  "first_mismatch": "" if not off else f"{off[0][0]}: frozen {off[0][1]}, file {off[0][2]}"})
    mismatches += [(q["id"],) + o for o in off] + [(q["id"], k, exp[k], "absent") for k in missing]

print(pd.DataFrame(agree).to_string(index=False))
print()
print(pd.DataFrame(rows).to_string(index=False))
print(f"\n{sum(1 for r in rows if r['result'] == 'FAIL')} of {len(rows)} questions disagree   "
      f"{sum(len(v) for v in RECOMPUTED.values())} values recomputed   "
      f"{sum(1 for q in QUESTIONS if q['expected_numeric'])} questions expected numbers")
print("mq-06 counts, event " + " / ".join(f"{symptom_ev[k]:,}" for k in (NO_HEAT, BOTH, NO_WATER))
      + "   quiet " + " / ".join(f"{symptom_qu[k]:,}" for k in (NO_HEAT, BOTH, NO_WATER)))
assert not mismatches, f"frozen expectations disagree with the artefacts: {mismatches}"

for q in QUESTIONS:
    ids = set(q["reference_record_ids"])
    if ids:
        assert ids <= set(feat.unique_key), f"{q['id']} names a ticket that is not in the features file"
        assert not ids & set(lab.unique_key), f"{q['id']} names a ticket that does have a label"
        assert ids == set(open_tickets.unique_key) and len(ids) == q["expected_numeric"]["unlabelled_open"]
        print(f"{q['id']}: {len(ids)} named records are all in the features file, all absent from the labels file, "
              f"and exactly the tickets with no outcome")


   id  values  exact_matches first_mismatch
mq-01       3              3               
mq-02      10             10               
mq-03       9              9               
mq-04      10             10               
mq-05       3              3               
mq-06      12             12               
mq-07       5              5               
mq-08       7              7               
mq-09       6              6               
mq-10       0              0               
mq-11       6              6               
mq-12       0              0               
mq-13       0              0               
mq-14       0              0               

            check result                            detail
 mq-01 — 3 values   PASS                         all exact
mq-02 — 10 values   PASS                         all exact
 mq-03 — 9 values   PASS                         all exact
mq-04 — 10 values   PASS                         all exact
 mq-05 — 3 values   PASS                    

## Gate 3: does each example actually test the boundary it is filed under?

Gate 3 checks that refusal reasons match their policy boundary: geography and outcome refusals name the relevant fields; missing-evidence and scope refusals name none. No reason may label an approved intake field as forbidden.


In [29]:
OUTCOME_FIELDS = ["closed_date", "status", "resolution_description", "resolution_action_updated_date", "due_date"]
GEO_FIELDS = ["incident_zip", "street_name", "latitude", "longitude", "cross_street_1", "cross_street_2",
              "intersection_street_1", "intersection_street_2"]
OVERCLAIM = ["sla", "caused", "predicts", "proves", "accuracy", "certainly", "because of", "target"]
INTAKE_NOFLAG = [f for f in INTAKE if f != "borough"]
by_class = {q["question_type"]: q for q in QUESTIONS if q["question_type"].startswith("refusal_")}
rows = []


def check(name, ok, detail):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


def names_in(text):
    return sorted(f for f in ALL_SOURCE if re.search(rf"\b{re.escape(f)}\b", text))


check("four refusal classes, one example each",
      len(by_class) == 4 == len([q for q in QUESTIONS if q["expected_response_type"] == "refuse"]),
      "  ".join(sorted(t.replace("refusal_", "") for t in by_class)))
check("no refusal reason calls an approved intake field forbidden", all(
    not set(names_in(q["expected_refusal_reason"])) & set(INTAKE_NOFLAG) for q in by_class.values()),
      f"only forbidden fields may be cited as forbidden; borough is exempt under its review flag "
      f"({len(INTAKE_NOFLAG)} unflagged intake fields)")
geo = names_in(by_class["refusal_precise_geography"]["expected_refusal_reason"])
out = names_in(by_class["refusal_outcome_field"]["expected_refusal_reason"])
check("the geography example tests the geography boundary", set(GEO_FIELDS) <= set(geo), f"{len(geo)} named")
check("the outcome example names every closure and outcome field", set(OUTCOME_FIELDS) <= set(out),
      f"{len(out)} named: {', '.join(out)}")
check("the scope example rests on scope, not on a missing column",
      not set(names_in(by_class["refusal_scope"]["expected_refusal_reason"])) & FORBIDDEN,
      "mq-14 names no forbidden field, so it cannot be mistaken for a field-boundary refusal")
check("mq-10 is the missing-evidence case, not an outcome case",
      ANSWER_ID["mq-10"]["question_type"] == "refusal_missing_evidence"
      and not set(names_in(ANSWER_ID["mq-10"]["expected_refusal_reason"])) & set(OUTCOME_FIELDS)
      and "weather" in ANSWER_ID["mq-10"]["question"].lower(),
      "weather causation is a boundary about evidence of conditions, not about a closure column")
check("the caveat that is not a refusal still answers", ANSWER_ID["mq-11"]["expected_response_type"] == "answer_with_caveat"
      and ANSWER_ID["mq-11"]["expected_numeric"] and not ANSWER_ID["mq-11"]["expected_refusal_reason"],
      "mq-11 must produce numbers plus the confound; refusing it would withhold an answerable figure")
check("every source names only the two artefacts it may read", all(
    q["source"]["artifact"] in {"features", "labels", "features + labels"} and q["source"]["local_query"]
    and q["source"]["published_in"] for q in QUESTIONS), f"{len(QUESTIONS)} source blocks filled in")
check("the wording bans every overclaim the design names", all(any(w in p for p in {x for q in QUESTIONS
                                                                                   for x in q["forbidden_phrasings"]})
                                                               for w in OVERCLAIM),
      "no SLA wording, no causation, no prediction, no proof, no false precision of certainty")
check("the two threshold questions both refuse SLA wording",
      all("sla" in ANSWER_ID[i]["forbidden_phrasings"] for i in ("mq-04", "mq-05")),
      "62.349333 h may be reported, never renamed a standard")
check("no question asks for a record set this notebook cannot pin", all(
    len(q["reference_record_ids"]) == 0 or q["question_type"] == "outcome_integrity" for q in QUESTIONS),
      "one question names records; the rest are aggregates, whose provenance is the filter")

gate3 = pd.DataFrame(rows)
print(gate3.to_string(index=False))
fails = int((gate3["result"] == "FAIL").sum())
print(f"\n{fails} of {len(gate3)} failed")
assert fails == 0, "a question is filed under a boundary it does not test"


                                                     check result                                                                                                               detail
                    four refusal classes, one example each   PASS                                                            missing_evidence  outcome_field  precise_geography  scope
no refusal reason calls an approved intake field forbidden   PASS only forbidden fields may be cited as forbidden; borough is exempt under its review flag (6 unflagged intake fields)
        the geography example tests the geography boundary   PASS                                                                                                              8 named
 the outcome example names every closure and outcome field   PASS                       5 named: closed_date, due_date, resolution_action_updated_date, resolution_description, status
 the scope example rests on scope, not on a missing column   PASS                    

## Write the question set and the expected-results file

Write `manager_questions_v1.jsonl` for the grader and `manager_questions_expected_v1.json` for reviewers. The expected-results file includes frozen and recomputed values, verdicts, and source hashes. Hash the bytes written to disk.


In [30]:
def as_json(obj):
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    raise TypeError(type(obj))


def write_text(path, text):
    body = text.encode("utf-8")
    path.write_bytes(body)
    return hashlib.sha256(body).hexdigest(), len(body)


OUT_DIR.mkdir(parents=True, exist_ok=True)
q_path, e_path = OUT_DIR / QUESTIONS_FILE, OUT_DIR / EXPECTED_FILE

expected_doc = {
    "questions_version": QUESTIONS_VERSION,
    "authored_before_recompute": True,
    "frozen_expectations": {q["id"]: q["expected_numeric"] for q in QUESTIONS if q["expected_numeric"]},
    "recomputed_from_artifacts": RECOMPUTED,
    "agreement": agree,
    "all_values_agree": not mismatches,
    "input_artifacts": {
        FEATURES_FILE: {"sha256": feat_side["artifact_sha256"], "bytes": feat_side["artifact_bytes"],
                        "rows": len(feat), "built_at": feat_side["built_at"]},
        LABELS_FILE: {"sha256": lab_side["artifact_sha256"], "bytes": lab_side["artifact_bytes"],
                      "rows": len(lab), "built_at": lab_side["built_at"], "label_threshold_hours": threshold_h},
    },
    "provenance": PROVENANCE,
    "question_types": {t: sum(1 for q in QUESTIONS if q["question_type"] == t) for t in sorted(REQUIRED_TYPES)},
    "response_types": {t: sum(1 for q in QUESTIONS if q["expected_response_type"] == t)
                       for t in ("answer", "answer_with_caveat", "refuse")},
    "refusal_classes": sorted(by_class),
    "known_limits": [
        "every expected value is a January 2024 measurement over two windows of one complaint type; none of them is "
        "a forecast or an accuracy claim",
        "the 47 ticket ids in mq-03 came from these artefacts, not an independent pull, so they pin a set rather "
        "than corroborate a number",
        "grading wording is string matching on this v1 set; it is not a language-model judge and inherits every "
        "phrasing choice made here",
        "the answers here are descriptive of past intake; no model was trained, no retrieval was run, no agent was "
        "asked",
        "shares are one decimal and the threshold six, matching how notebook 03 published them; a grader must not "
        "re-round them differently",
        "borough appears in answers under the open fairness/proxy review notebook 03 flagged",
    ],
}
written = {
    q_path: write_text(q_path, "".join(json.dumps(q, ensure_ascii=False) + "\n" for q in QUESTIONS)),
    e_path: write_text(e_path, json.dumps(expected_doc, indent=2, default=as_json)),
}
for path, (sha, size) in written.items():
    print(f"{path.name:<42} {size:>9,} bytes   sha256 {sha[:16]}")
print(f"\n{len(QUESTIONS)} question records | {len(expected_doc['frozen_expectations'])} questions with numbers | "
      f"{len(expected_doc['recomputed_from_artifacts'])} recomputed | "
      f"{sum(len(v) for v in RECOMPUTED.values())} values | refusals {expected_doc['response_types']['refuse']}")


manager_questions_v1.jsonl                    16,405 bytes   sha256 823f660ea8d36fb7
manager_questions_expected_v1.json            11,032 bytes   sha256 2d6ad5f5fdf7fe33

14 question records | 10 questions with numbers | 10 recomputed | 71 values | refusals 4


## Prove the files on disk are the files we wrote

Read both files back and compare them with the in-memory records. Verify the sidecar hashes against the saved bytes.


In [31]:
from_disk = [json.loads(line) for line in q_path.read_text(encoding="utf-8").splitlines() if line.strip()]
expected_disk = json.loads(e_path.read_text(encoding="utf-8"))
rows = []


def check(name, ok, detail):
    rows.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})


check("14 records read back", len(from_disk) == 14, f"{len(from_disk)} lines")
check("read-back records equal the authored records exactly", from_disk == QUESTIONS,
      "field order, values and ids all identical")
check("every read-back record has the ten fields", all(list(q) == FIELDS for q in from_disk), "no schema drift")
check("the expected file records a verdict per question", len(expected_disk["agreement"]) == 14
      and expected_disk["all_values_agree"], f"all_values_agree={expected_disk['all_values_agree']}")
check("the two files hold the same expectations", all(
    expected_disk["frozen_expectations"][q["id"]] == q["expected_numeric"]
    for q in from_disk if q["expected_numeric"]), "one truth, written twice on purpose, and compared")
check("recorded sha256 is each file's own bytes", all(
    hashlib.sha256(p.read_bytes()).hexdigest() == sha for p, (sha, _) in written.items()),
      "nothing edited the files between writing and checking")
check("input artefact hashes carried into the expected file",
      expected_disk["input_artifacts"][FEATURES_FILE]["sha256"] == feat_side["artifact_sha256"]
      and expected_disk["input_artifacts"][LABELS_FILE]["sha256"] == lab_side["artifact_sha256"],
      "a reader can tell which January files these expectations belong to")
check("notebook 03's threshold travelled unchanged",
      expected_disk["input_artifacts"][LABELS_FILE]["label_threshold_hours"] == lab_side["label_threshold_hours"],
      f"{threshold_h!r}")

g = pd.DataFrame(rows)
print(g.to_string(index=False))
fails = int((g["result"] == "FAIL").sum())
print(f"\n{fails} of {len(g)} failed")
assert fails == 0, "what is on disk is not what was checked"


                                               check result                                                             detail
                                14 records read back   PASS                                                           14 lines
read-back records equal the authored records exactly   PASS                          field order, values and ids all identical
           every read-back record has the ten fields   PASS                                                    no schema drift
    the expected file records a verdict per question   PASS                                              all_values_agree=True
            the two files hold the same expectations   PASS                  one truth, written twice on purpose, and compared
            recorded sha256 is each file's own bytes   PASS              nothing edited the files between writing and checking
input artefact hashes carried into the expected file   PASS a reader can tell which January files these expecta

## Provenance: sidecars

Each `.meta.json` records inputs, limitations, and the artifact hash. `network_used` reflects the notebook's source and import checks; it is not a packet capture.


In [32]:
built_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
common = {
    "built_at": built_at,
    "built_by": THIS_NOTEBOOK,
    "questions_version": QUESTIONS_VERSION,
    "network_used": False,
    "http_requests_sent": 0,
    "network_guard": {"libraries_checked": BANNED_MODULES, "present_in_kernel": imported,
                      "meaning": "no HTTP, model, vector-store or tracing client is imported here; an import-level "
                                 "check, not a packet capture"},
    "input_artifacts": {
        FEATURES_FILE: {"sha256": feat_side["artifact_sha256"], "rows": len(feat), "windows": feat_side["windows"],
                        "model_inputs": feat_side["model_inputs"]},
        LABELS_FILE: {"sha256": lab_side["artifact_sha256"], "rows": len(lab),
                      "label_threshold_hours": threshold_h, "label_rule": lab_side["label_rule"]},
    },
    "carried_from_notebook_03": {"label_threshold_hours": threshold_h, "review_flags": feat_side["review_flags"],
                                 "deliberately_excluded": feat_side["deliberately_excluded"],
                                 "note": "notebook 03's files and threshold are read-only inputs here; nothing in "
                                         "this notebook rewrote them"},
    "forbidden_fields_in_scope": len(FORBIDDEN),
    "questions": len(QUESTIONS),
    "expected_values": sum(len(v) for v in RECOMPUTED.values()),
    "all_values_agree": not mismatches,
}
sidecars = {
    q_path: {**common, "columns": FIELDS, "question_types": expected_doc["question_types"],
             "response_types": expected_doc["response_types"], "refusal_classes": expected_doc["refusal_classes"],
             "known_limits": expected_doc["known_limits"]},
    e_path: {**common, "keys": list(expected_doc),
             "known_limits": ["this file is the run's verdict, not a second source of truth: the questions file holds "
                              "the expectations, this one holds them beside what was measured",
                              "values are compared at the precision they were authored at",
                              "provenance cites cell indexes of finished runs; re-running notebook 01 or 03 could "
                              "move those indexes and would break the citation, not the number",
                              *expected_doc["known_limits"][:3]]},
}
for path, payload in sidecars.items():
    meta_path = path.with_name(path.name.replace(path.suffix, ".meta.json"))
    sha, size = write_text(meta_path, json.dumps(payload, indent=2, default=as_json))
    print(f"{meta_path.name:<46} {size:>7,} bytes   {len(payload)} keys   sha256 {sha[:16]}")
print(f"\nnetwork_used=False recorded in both sidecars   built_at {built_at}")


manager_questions_v1.meta.json                   4,138 bytes   17 keys   sha256 bf7f410dac3dc18b
manager_questions_expected_v1.meta.json          3,690 bytes   14 keys   sha256 c8f648ccc7e874ec

network_used=False recorded in both sidecars   built_at 2026-09-24T22:58:09Z


## What this evaluation set can and cannot claim

Fill nothing in below from memory: every figure here is this run's own output.

**Can claim**
- Fourteen manager questions whose `<N>` expected values were frozen before this run and then reproduced from the two
  notebook 03 artefacts by a different code path — `<agreement>`, with mq-06's symptom counts matching notebook 01's
  own per-window table exactly.
- That four of them must be refused, and that each refusal tests a different boundary: absent weather evidence,
  forbidden precise geography, forbidden outcome fields, and a population these files do not cover. The scope refusal
  names no field, which is what keeps it a scope refusal.
- That one question is an answer with a caveat rather than a refusal, and that the set enforces the difference
  instead of relying on a reader to keep it straight.
- That `62.349333 h` appears here only as a fitted local threshold with its history attached, never as a service
  level.

**Cannot claim**
- That any system has been tested. Nothing was generated, so there is no answer to grade yet and no score exists.
- That a string match on `acceptable_phrasings` is a fair judge of prose. It is a v1 grader with a v1 grader's
  rigidity: a correct answer phrased another way fails here before it fails in production.
- That fourteen questions cover what managers ask. They cover five agreed categories, in one complaint type, in one
  January.
- That the event/quiet contrast is causal. mq-10 refuses exactly that, and refusing it does not make the remaining
  numbers predictive.
- That anything here generalises past 2024-01-15 to 2024-01-22 and 2024-01-24 to 2024-01-30, `HEAT/HOT WATER`, and
  closed tickets only.

**What notebook 05 may take from here**
`manager_questions_v1.jsonl` as reference inputs and reference outputs, and the agreement table as evidence that the
expectations were authored rather than discovered. Refusals are graded by reason class, not by exact sentence.
Track 1's periods, `T_SLOW`, `H_MAT` and the 2023 pull stay in notebook 05 — nothing in this set depends on them.


## How to review it, and how to run it

Read it in this order; four places are enough to catch a bad number.

1. **The question text and the frozen literals** — the three authoring cells. This is the part that must be
   reviewable without running anything: a value you disagree with is changed here, not downstream.
2. **Gate 2.** One row per question: `values / exact_matches / first_mismatch`. Fourteen rows agreeing is the whole
   point of the notebook.
3. **Gate 3.** Whether each refusal really names the fields it claims, and whether mq-10 and mq-14 still name none.
4. **The citations cell.** It opens notebooks 01 and 03, reads their stored outputs and searches for the figures. A
   wrong cell index or a misquoted string fails there.

To run it: the same way you ran notebooks 01–03, same kernel, Restart & Run All. It writes two files and two
sidecars into `data/evaluation/` and sends no request. Notebook 03's artefacts are opened read-only, and the
integrity gate prints their hashes so you can see they still match the sidecars notebook 03 left behind.

Deliberately absent: no model call, no vector index, no RAGAS, no LangSmith, no classifier, no time split. That is
Track 1, and Track 1 needs the 2023 pull this notebook was not allowed to make.
